In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import glob
import threading

# Global variables
drawing = False  # True if the mouse is pressed
points = []  # List to store points for freehand drawing
label_options = os.listdir('/home/natsu/MechAI/Dataset')
img = None  # Global variable for the current image
img_copy = None  # Global variable for the image copy
selected_label = None  # Global variable to store the selected label
processing_threads = []  # List to hold processing threads


def draw_freehand(event, x, y, flags, param):
    global drawing, points, img, img_copy

    if event == cv2.EVENT_LBUTTONDOWN:
        drawing = True
        points = [(x, y)]
        img_copy = img.copy()

    elif event == cv2.EVENT_MOUSEMOVE:
        if drawing:
            img_copy = img.copy()
            points.append((x, y))
            for i in range(1, len(points)):
                cv2.line(img_copy, points[i - 1], points[i], (0, 255, 0), 2)
            display_label_options(img_copy)
            cv2.imshow("Image", img_copy)

    elif event == cv2.EVENT_LBUTTONUP:
        drawing = False
        points.append((x, y))
        cv2.line(img_copy, points[0], points[-1], (0, 255, 0), 2)
        display_label_options(img_copy)
        cv2.imshow("Image", img_copy)


def display_label_options(image):
    global selected_label
    height, width = image.shape[:2]

    label_area_width = 300
    font = cv2.FONT_HERSHEY_SIMPLEX
    y_offset = 30

    for i, label in enumerate(label_options):
        color = (255, 255, 255)
        if selected_label == label:
            color = (0, 255, 0)
        cv2.putText(image, f"{i+1}: {label}", (width - label_area_width + 10, y_offset), font, 1, color, 2, cv2.LINE_AA)
        y_offset += 40


def save_cropped_image(img, save_dir):
    global points, selected_label
    if selected_label is None:
        print("No label selected, skipping this image.")
        return

    mask = np.full(img.shape[:2], cv2.GC_PR_BGD, dtype=np.uint8)
    if len(points) > 2:
        points_array = np.array(points, dtype=np.int32)
        cv2.fillPoly(mask, [points_array], cv2.GC_PR_FGD)

        bgd_model = np.zeros((1, 65), np.float64)
        fgd_model = np.zeros((1, 65), np.float64)
        rect = cv2.boundingRect(points_array)
        cv2.grabCut(img, mask, rect, bgd_model, fgd_model, 5, cv2.GC_INIT_WITH_MASK)

        mask2 = np.where((mask == cv2.GC_FGD) | (mask == cv2.GC_PR_FGD), 1, 0).astype('uint8')
        result = img * mask2[:, :, np.newaxis]

        x, y, w, h = cv2.boundingRect(points_array)
        cropped_img = result[y:y+h, x:x+w]

        save_path = os.path.join(save_dir, f"{selected_label}.png")
        cv2.imwrite(save_path, cropped_img)
        print(f"Cropped image saved as: {save_path}")
    else:
        print("Shape not drawn. Please draw a complete shape next time.")


def crop_and_label_object(img_input, save_dir):
    global img, img_copy, selected_label, points
    img = img_input
    img_copy = img.copy()
    selected_label = None
    points.clear()

    cv2.imshow("Image", img_copy)
    cv2.setMouseCallback("Image", draw_freehand)
    display_label_options(img_copy)

    while True:
        key = cv2.waitKey(1) & 0xFF
        if ord('1') <= key <= ord('5'):
            selected_label = label_options[key - ord('1')]
            print(f"Label '{selected_label}' selected.")
            break
        if key == ord('q'):
            print("Quitting...")
            break

    cv2.destroyAllWindows()


def process_image_in_thread(img, save_dir):
    save_cropped_image(img, save_dir)


def process_images_in_folder(image_folder, save_dir):
    global processing_threads
    image_files = glob.glob(os.path.join(image_folder, "*.jpg")) + \
                  glob.glob(os.path.join(image_folder, "*.png")) + \
                  glob.glob(os.path.join(image_folder, "*.jpeg"))

    for image_path in image_files:
        print(f"Processing image: {image_path}")
        img = cv2.imread(image_path)
        if img is not None:
            crop_and_label_object(img, save_dir)
            thread = threading.Thread(target=process_image_in_thread, args=(img.copy(), save_dir))
            thread.start()
            processing_threads.append(thread)
        else:
            print(f"Failed to load image: {image_path}")

    for thread in processing_threads:
        thread.join()

    print("All images processed.")


# Example usage
process_images_in_folder('/home/natsu/MechAI/Dataset/disk_rotor', '/home/natsu/MechAI/Dataset/disk_rotor_2')


Processing image: /home/natsu/MechAI/Dataset/disk_rotor/disk_rotor_132.png


QObject::moveToThread: Current thread (0x26c94e0) is not the object's thread (0x2c529e0).
Cannot move to target thread (0x26c94e0)

QObject::moveToThread: Current thread (0x26c94e0) is not the object's thread (0x2c529e0).
Cannot move to target thread (0x26c94e0)

QObject::moveToThread: Current thread (0x26c94e0) is not the object's thread (0x2c529e0).
Cannot move to target thread (0x26c94e0)

QObject::moveToThread: Current thread (0x26c94e0) is not the object's thread (0x2c529e0).
Cannot move to target thread (0x26c94e0)

QObject::moveToThread: Current thread (0x26c94e0) is not the object's thread (0x2c529e0).
Cannot move to target thread (0x26c94e0)

QObject::moveToThread: Current thread (0x26c94e0) is not the object's thread (0x2c529e0).
Cannot move to target thread (0x26c94e0)

QObject::moveToThread: Current thread (0x26c94e0) is not the object's thread (0x2c529e0).
Cannot move to target thread (0x26c94e0)

QObject::moveToThread: Current thread (0x26c94e0) is not the object's thread

Label 'break_caliper' selected.
Processing image: /home/natsu/MechAI/Dataset/disk_rotor/disk_rotor_47.png


QObject::moveToThread: Current thread (0x26c94e0) is not the object's thread (0x2c529e0).
Cannot move to target thread (0x26c94e0)

QObject::moveToThread: Current thread (0x26c94e0) is not the object's thread (0x2c529e0).
Cannot move to target thread (0x26c94e0)

QObject::moveToThread: Current thread (0x26c94e0) is not the object's thread (0x2c529e0).
Cannot move to target thread (0x26c94e0)

QObject::moveToThread: Current thread (0x26c94e0) is not the object's thread (0x2c529e0).
Cannot move to target thread (0x26c94e0)

QObject::moveToThread: Current thread (0x26c94e0) is not the object's thread (0x2c529e0).
Cannot move to target thread (0x26c94e0)

QObject::moveToThread: Current thread (0x26c94e0) is not the object's thread (0x2c529e0).
Cannot move to target thread (0x26c94e0)

QObject::moveToThread: Current thread (0x26c94e0) is not the object's thread (0x2c529e0).
Cannot move to target thread (0x26c94e0)

QObject::moveToThread: Current thread (0x26c94e0) is not the object's thread

Label 'break_caliper' selected.
Processing image: /home/natsu/MechAI/Dataset/disk_rotor/disk_rotor_108.png


...

Label 'coil_spring' selected.
Processing image: /home/natsu/MechAI/Dataset/disk_rotor/disk_rotor_170.png


QObject::moveToThread: Current thread (0x26c94e0) is not the object's thread (0x2c529e0).
Cannot move to target thread (0x26c94e0)

QObject::moveToThread: Current thread (0x26c94e0) is not the object's thread (0x2c529e0).
Cannot move to target thread (0x26c94e0)

QObject::moveToThread: Current thread (0x26c94e0) is not the object's thread (0x2c529e0).
Cannot move to target thread (0x26c94e0)

QObject::moveToThread: Current thread (0x26c94e0) is not the object's thread (0x2c529e0).
Cannot move to target thread (0x26c94e0)

QObject::moveToThread: Current thread (0x26c94e0) is not the object's thread (0x2c529e0).
Cannot move to target thread (0x26c94e0)

QObject::moveToThread: Current thread (0x26c94e0) is not the object's thread (0x2c529e0).
Cannot move to target thread (0x26c94e0)

QObject::moveToThread: Current thread (0x26c94e0) is not the object's thread (0x2c529e0).
Cannot move to target thread (0x26c94e0)

QObject::moveToThread: Current thread (0x26c94e0) is not the object's thread

Quitting...
No label selected, skipping this image.
Processing image: /home/natsu/MechAI/Dataset/disk_rotor/disk_rotor_261.png


QObject::moveToThread: Current thread (0x26c94e0) is not the object's thread (0x2c529e0).
Cannot move to target thread (0x26c94e0)

QObject::moveToThread: Current thread (0x26c94e0) is not the object's thread (0x2c529e0).
Cannot move to target thread (0x26c94e0)

QObject::moveToThread: Current thread (0x26c94e0) is not the object's thread (0x2c529e0).
Cannot move to target thread (0x26c94e0)

QObject::moveToThread: Current thread (0x26c94e0) is not the object's thread (0x2c529e0).
Cannot move to target thread (0x26c94e0)

QObject::moveToThread: Current thread (0x26c94e0) is not the object's thread (0x2c529e0).
Cannot move to target thread (0x26c94e0)

QObject::moveToThread: Current thread (0x26c94e0) is not the object's thread (0x2c529e0).
Cannot move to target thread (0x26c94e0)

QObject::moveToThread: Current thread (0x26c94e0) is not the object's thread (0x2c529e0).
Cannot move to target thread (0x26c94e0)

QObject::moveToThread: Current thread (0x26c94e0) is not the object's thread

Cropped image saved as: /home/natsu/MechAI/Dataset/disk_rotor_2/None.png
